In [130]:
import os
import io
import json
import numpy as np
from PIL import Image

import torch

# If you run notebook from project root, this is usually enough.
# If not, set PROJECT_ROOT explicitly.
import sys
PROJECT_ROOT = os.path.abspath("..")  # change if needed
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from models.multi_expert_v2 import create_multi_expert_v2

ORION_CHANNEL_NAMES = [
    'Hoechst', 'CD31', 'CD45', 'CD68', 'CD4', 'FOXP3',
    'CD8a', 'CD45RO', 'CD20', 'PD-L1', 'CD3e', 'CD163',
    'E-cadherin', 'PD-1', 'Ki67', 'Pan-CK', 'SMA'
]

# ImageNet normalization
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)


In [131]:
def get_device(device_preference: str = "cuda") -> torch.device:
    if device_preference == "cuda" and torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

device = get_device("cuda")
device


device(type='cuda')

In [3]:
def infer_stpath_dim_from_checkpoint(checkpoint: dict, fallback: int = 38984) -> int:
    stpath_key = None
    for k in checkpoint["model_state_dict"].keys():
        if ("stpath_expert" in k) and ("emb_to_kv.weight" in k):
            stpath_key = k
            break
    if stpath_key is None:
        return fallback
    # emb_to_kv weight shape: [feat_dim * num_tokens * 2, emb_dim] -> emb_dim is shape[1]
    return checkpoint["model_state_dict"][stpath_key].shape[1]


def load_orion_model(checkpoint_path: str, gigatime_weights: str, device: torch.device) -> torch.nn.Module:
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    cfg = ckpt["config"]

    uni_dim   = 0 if cfg.get("disable_uni", False)   else 1536
    conch_dim = 0 if cfg.get("disable_conch", False) else 768
    stpath_dim = 0 if cfg.get("disable_stpath", False) else infer_stpath_dim_from_checkpoint(ckpt, fallback=38984)

    model = create_multi_expert_v2(
        weights_path=gigatime_weights,
        freeze_gigatime=cfg.get("freeze_gigatime", True),
        num_heads=cfg.get("num_heads", 4),
        num_tokens=cfg.get("num_tokens", 8),
        dropout=cfg.get("dropout", 0.1),
        use_cross_expert=cfg.get("use_cross_expert", False),
        use_dynamic_gating=cfg.get("use_dynamic_gating", False),
        use_multiscale_film=cfg.get("use_multiscale_film", False),
        uni_dim=uni_dim,
        conch_dim=conch_dim,
        stpath_dim=stpath_dim,
        out_channels=17,
    )

    model.load_state_dict(ckpt["model_state_dict"])
    model.to(device).eval()

    print(f"Loaded checkpoint: {checkpoint_path}")
    print(f"  epoch: {ckpt.get('epoch', '?')}")
    print(f"  Experts: UNI={uni_dim}, CONCH={conch_dim}, STPath={stpath_dim}")
    return model


In [4]:
def preprocess_he_image(img_path: str, input_size: int = 512) -> torch.Tensor:
    img = Image.open(img_path).convert("RGB")
    img = img.resize((input_size, input_size), Image.BILINEAR)
    arr = np.asarray(img).astype(np.float32) / 255.0
    arr = (arr - MEAN) / STD
    arr = arr.transpose(2, 0, 1)  # HWC -> CHW
    return torch.from_numpy(arr).unsqueeze(0)  # [1,3,H,W]


def load_report_embeddings(emb_dir: str, name: str, device: torch.device):
    """
    Expected files:
      {name}_univ2.pkl
      {name}_conch.pkl
      {name}_stpath.pkl
    """
    uni_path   = os.path.join(emb_dir, f"{name}_univ2.pkl")
    conch_path = os.path.join(emb_dir, f"{name}_conch.pkl")
    stp_path   = os.path.join(emb_dir, f"{name}_stpath.pkl")

    emb_uni = torch.load(uni_path, map_location=device, weights_only=False).squeeze(0)     # [1536]
    emb_conch = torch.load(conch_path, map_location=device, weights_only=False).squeeze(0) # [768]
    emb_stpath = torch.load(stp_path, map_location=device, weights_only=False)             # [38984] or similar

    return emb_uni.unsqueeze(0), emb_conch.unsqueeze(0), emb_stpath.unsqueeze(0)


In [5]:
@torch.no_grad()
def predict_mean_expression(
    model: torch.nn.Module,
    img_tensor: torch.Tensor,
    emb_uni: torch.Tensor,
    emb_conch: torch.Tensor,
    emb_stpath: torch.Tensor,
    device: torch.device,
) -> np.ndarray:
    img_tensor = img_tensor.to(device, non_blocking=True)
    emb_uni = emb_uni.to(device, non_blocking=True)
    emb_conch = emb_conch.to(device, non_blocking=True)
    emb_stpath = emb_stpath.to(device, non_blocking=True)

    pred, _ = model(img_tensor, emb_uni, emb_conch, emb_stpath)  # [1,17,H,W]
    mean_expr = pred[0].mean(dim=(1, 2)).detach().cpu().numpy()  # [17]
    return mean_expr


def rank_biomarkers(mean_expr: np.ndarray):
    order = np.argsort(-mean_expr)
    return [(ORION_CHANNEL_NAMES[i], float(mean_expr[i])) for i in order]


In [7]:
import pandas as pd

def collect_pngs(slide_dir: str, sets_prefix: str | None = None):
    """
    Returns list of (set_name, img_path, name)
    """
    sets = sorted([d for d in os.listdir(slide_dir) if os.path.isdir(os.path.join(slide_dir, d))])
    items = []
    for set_name in sets:
        if sets_prefix is not None and not set_name.startswith(sets_prefix):
            continue
        set_dir = os.path.join(slide_dir, set_name)
        pngs = sorted([f for f in os.listdir(set_dir) if f.endswith(".png")])
        for f in pngs:
            name = f[:-4]
            items.append((set_name, os.path.join(set_dir, f), name))
    return items


In [77]:
import gzip
import io
from PIL import Image
import scanpy as sc

slide_info = ['GSM8797975_S11_SpT',
 'GSM8797978_S4_SpT',
 'GSM8797974_S10_SpT',
 'GSM8797980_S6_SpT',
 'GSM8797979_S5_SpT',
 'GSM8797977_S3_SpT',
 'GSM8797983_S9_SpT',
 'GSM8797973_S1_SpT',
 'GSM8797982_S8_SpT',
 'GSM8797981_S7_SpT',
 'GSM8797976_S15_SpT']

In [15]:
import pandas as pd

In [20]:
def preprocess_he_image(img, input_size: int = 512) -> torch.Tensor:
    img = img.resize((input_size, input_size), Image.BILINEAR)
    arr = np.asarray(img).astype(np.float32) / 255.0
    arr = (arr - MEAN) / STD
    arr = arr.transpose(2, 0, 1)  # HWC -> CHW
    return torch.from_numpy(arr).unsqueeze(0)  # [1,3,H,W]

def run_orion_inference(
    model: torch.nn.Module,
    slide_dir: str,
    emb_dir: str,
    device: torch.device,
    input_size: int = 512,
    max_images: int | None = None,
    verbose: bool = True,
    sample_info = 'None'
):
    """
    Returns:
      df_long: one row per (image, biomarker) with mean_expression + rank
      results_dict: JSON-serializable grouped structure (set_name -> list of images)
    """
    items = collect_pngs(slide_dir)
    if max_images is not None:
        items = items[:max_images]

    # Optional tqdm
    try:
        from tqdm.auto import tqdm
        it = tqdm(items, desc="Inference", total=len(items))
    except Exception:
        it = items

    rows = []
    results_dict = {}
    key = sample_info
#     key = "GSM8797973_S1_SpT"
    emb_uni = torch.load(f"../../allemb_cscc/{key}_univ2.pkl")
    emb_conch = torch.load(f"../../allemb_cscc/{key}_conch.pkl")
    emb_stpath = pd.read_pickle(f"../../allemb_cscc/{key}_stpath.pkl")['pred']
    emb_stpath = torch.FloatTensor(emb_stpath)
    adata = sc.read_h5ad(f"../../cscc_info_mixtime/gsminfo_out/{key}_out.h5ad")
    image_info = Image.open(f"/home/tl688/zhao_project/GigaTIME/cscc_info_mixtime/gsminfo_out/{key}_tissue_hires_image.png").convert('RGB')
    mean_expr_list = []
    for idx,loc in enumerate(adata.obsm['X_loc']):
        image_new = image_info.crop((int(loc[0]) - 256, int(loc[1]) - 256, int(loc[0]) + 256, int(loc[1]) + 256))
        img_tensor = preprocess_he_image(image_new, input_size=input_size)
        mean_expr = predict_mean_expression(model, img_tensor, emb_uni[idx,:], emb_conch[idx,:], emb_stpath[idx,:], device)
        mean_expr_list.append(mean_expr)
    return np.array(mean_expr_list)

In [ ]:
checkpoint_path = "../orion_gene_mixpearson/best_model.pth"
gigatime_weights = "../e48822b5419308cf918ae920239408d7b33327fa/model.pth"
slide_dir = "../../cscc_info_mixtime/"
emb_dir = "./../cscc_info_mixtime/"

device = get_device("cuda")
model = load_orion_model(checkpoint_path, gigatime_weights, device)

for key in slide_info:
    out1 = run_orion_inference(
        model=model,
        slide_dir=slide_dir,
        emb_dir=emb_dir,
        device=device,
        input_size=512,
        max_images=None,   # e.g. 10 for a quick test
        verbose=False,
        sample_info = key
    )
    
    np.save(f"../{key}_mixtime_out.npy", out1)


Loaded pretrained weights from ../e48822b5419308cf918ae920239408d7b33327fa/model.pth
Loaded checkpoint: ../orion_gene_mixpearson/best_model.pth
  epoch: 8
  Experts: UNI=1536, CONCH=768, STPath=38984


Inference:   0%|          | 0/11 [00:00<?, ?it/s]

Inference:   0%|          | 0/11 [00:00<?, ?it/s]

Inference:   0%|          | 0/11 [00:00<?, ?it/s]

Inference:   0%|          | 0/11 [00:00<?, ?it/s]

...

Inference:   0%|          | 0/11 [00:00<?, ?it/s]

Inference:   0%|          | 0/11 [00:00<?, ?it/s]

Inference:   0%|          | 0/11 [00:00<?, ?it/s]

Inference:   0%|          | 0/11 [00:00<?, ?it/s]

Inference:   0%|          | 0/11 [00:00<?, ?it/s]